In [1]:
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
import matplotlib.pyplot as plt
import pickle

In [2]:
from tensorflow.compat.v1 import ConfigProto
import os
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.9
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

In [3]:
from load_network import load_VGG
from train import WarmUpCosine, CustomWeightDecaySGD, RSquared
from save_load import load_noise_if_exists, save_layer_outputs_and_labels, load_layer_outputs_and_labels
from weights_bias_reg import load_wb_if_exists
from evaluate_reg import evalu_stream_main_selected, evalu_select, eval_acc_select_list_single_thresholds, evalu_prepare, compute_stats

In [4]:
SAVE_PATH = "utkface_12k_split.npz" 
def load_dataset_if_exists(path=SAVE_PATH):
    """
    Load if file exists, otherwise return None.
    """
    if os.path.exists(path):
        data = np.load(path)
        print("✔ Cached data detected, loading directly")
        return (data["x_train"], data["y_train"],
                data["x_test"], data["y_test"])
    return None

In [5]:
x_train, y_train, x_test, y_test = load_dataset_if_exists()

✔ 检测到缓存数据，已直接加载


In [6]:
model=load_VGG()

In [7]:
pred_model = tf.keras.Model(
    inputs=model.get_layer("re_lu_9").output,
    outputs=model.output
)

In [8]:
l_label = [2,6,10,13,17,20,24,27,32,35]

In [9]:
layer_list = [model.layers[l].name for l in l_label]

In [10]:
NOISE_DIR = "./noise/"
save_layer_outputs_and_labels(model, x_train, y_train, layer_list, save_dir="D:/Data/VGG-11/layer_cache/base")
for i in range(10):
    NOISE_I_DIR = NOISE_DIR + str(i)
    x_gauss, x_salt, x_move, x_occ = load_noise_if_exists(NOISE_I_DIR)
    save_layer_outputs_and_labels(model, x_gauss, y_train, layer_list, save_dir="D:/Data/VGG-11/layer_cache/gauss/"+str(i))
    save_layer_outputs_and_labels(model, x_salt, y_train, layer_list, save_dir="D:/Data/VGG-11/layer_cache/salt/"+str(i))
    save_layer_outputs_and_labels(model, x_move, y_train, layer_list, save_dir="D:/Data/VGG-11/layer_cache/move/"+str(i))
    save_layer_outputs_and_labels(model, x_occ, y_train, layer_list, save_dir="D:/Data/VGG-11/layer_cache/occ/"+str(i))

[SAVE] Generating layer outputs for: D:/Data/VGG-11/layer_cache/base
[Saved] re_lu: outputs (20000, 32768), labels (20000,)
[Saved] re_lu_1: outputs (20000, 8192), labels (20000,)
[Saved] re_lu_2: outputs (20000, 4096), labels (20000,)
[Saved] re_lu_3: outputs (20000, 4096), labels (20000,)
[Saved] re_lu_4: outputs (20000, 2048), labels (20000,)
[Saved] re_lu_5: outputs (20000, 2048), labels (20000,)
[Saved] re_lu_6: outputs (20000, 1024), labels (20000,)
[Saved] re_lu_7: outputs (20000, 1024), labels (20000,)
[Saved] re_lu_8: outputs (20000, 256), labels (20000,)
[Saved] re_lu_9: outputs (20000, 256), labels (20000,)
[SAVE] Generating layer outputs for: D:/Data/VGG-11/layer_cache/gauss/0
[Saved] re_lu: outputs (20000, 32768), labels (20000,)
[Saved] re_lu_1: outputs (20000, 8192), labels (20000,)
[Saved] re_lu_2: outputs (20000, 4096), labels (20000,)
[Saved] re_lu_3: outputs (20000, 4096), labels (20000,)
[Saved] re_lu_4: outputs (20000, 2048), labels (20000,)
[Saved] re_lu_5: output

In [11]:
for i in range(5):
    path = "./attack/" + str(i)
    ATTACK_DIR = os.path.join(path, "VGG_pgd.npy")
    x_attack = np.load(ATTACK_DIR)
    save_layer_outputs_and_labels(model, x_attack, y_train, layer_list, save_dir="D:/Data/VGG-11/layer_cache/attack/"+str(i))

[SAVE] Generating layer outputs for: D:/Data/VGG-11/layer_cache/attack/0
[Saved] re_lu: outputs (20000, 32768), labels (20000,)
[Saved] re_lu_1: outputs (20000, 8192), labels (20000,)
[Saved] re_lu_2: outputs (20000, 4096), labels (20000,)
[Saved] re_lu_3: outputs (20000, 4096), labels (20000,)
[Saved] re_lu_4: outputs (20000, 2048), labels (20000,)
[Saved] re_lu_5: outputs (20000, 2048), labels (20000,)
[Saved] re_lu_6: outputs (20000, 1024), labels (20000,)
[Saved] re_lu_7: outputs (20000, 1024), labels (20000,)
[Saved] re_lu_8: outputs (20000, 256), labels (20000,)
[Saved] re_lu_9: outputs (20000, 256), labels (20000,)
[SAVE] Generating layer outputs for: D:/Data/VGG-11/layer_cache/attack/1
[Saved] re_lu: outputs (20000, 32768), labels (20000,)
[Saved] re_lu_1: outputs (20000, 8192), labels (20000,)
[Saved] re_lu_2: outputs (20000, 4096), labels (20000,)
[Saved] re_lu_3: outputs (20000, 4096), labels (20000,)
[Saved] re_lu_4: outputs (20000, 2048), labels (20000,)
[Saved] re_lu_5: o

In [12]:
CACHE_DIR = "./VGG-11/w_and_b_cache"

In [13]:
eva_w,eva_b = load_wb_if_exists(y_train, layer_list, CACHE_DIR, save_dir="D:/Data/VGG-11/layer_cache/base")

In [14]:
threshold_list, Y_list = evalu_prepare(y_train, n=9)

In [15]:
select_list = evalu_select(layer_list, y_train, eva_w, eva_b, pred_model=pred_model, save_dir="D:/Data/VGG-11/layer_cache/base")

In [16]:
base = evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir="D:/Data/VGG-11/layer_cache/base")
base_final = eval_acc_select_list_single_thresholds(model, x_train, 'train', select_list, threshold_list) 
base = np.concatenate((base,base_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.88425941 0.71393082 0.65862808 0.60594888 0.62623677 0.52566628
 0.58168523 0.68405011 0.86831332]
Layer 1
accuracy: [0.92734973 0.79877934 0.68892602 0.63067496 0.68740743 0.65015866
 0.64396192 0.69400338 0.86023371]
Layer 2
accuracy: [0.93063163 0.79673541 0.69405008 0.65696781 0.71344042 0.69959519
 0.62137501 0.72181915 0.80079388]
Layer 3
accuracy: [0.95996603 0.83950417 0.72125373 0.6355908  0.72346921 0.72781424
 0.64513807 0.71815384 0.80674285]
Layer 4
accuracy: [0.96275817 0.87547813 0.7711519  0.71684381 0.76789047 0.80843719
 0.78322981 0.84372451 0.89314449]
Layer 5
accuracy: [0.96319713 0.87821805 0.75923005 0.73641367 0.78200761 0.82526938
 0.79213797 0.84587234 0.87920475]
Layer 6
accuracy: [0.97404626 0.95757748 0.86144471 0.8304428  0.8401431  0.87543387
 0.88568148 0.90506334 0.93253998]
Layer 7
accuracy: [0.97463338 0.94685121 0.82632831 0.82255379 0.88101099 0.90678903
 0.90875179 0.94172406 0.93793881]
Layer 8
accuracy: [0.98519134 0.9909095 

In [17]:
compute_stats(base)

(array([[0.75227277, 0.58595064, 0.71134955],
        [0.80501836, 0.65608035, 0.732733  ],
        [0.80713904, 0.69000114, 0.71466268],
        [0.84024131, 0.69562475, 0.72334492],
        [0.86979607, 0.76439049, 0.84003294],
        [0.86688174, 0.78123022, 0.83907169],
        [0.93102282, 0.84867326, 0.9077616 ],
        [0.91593763, 0.87011794, 0.92947155],
        [0.98329482, 0.90465806, 0.93514458],
        [0.98157012, 0.91840883, 0.97018445],
        [1.        , 1.        , 1.        ]]),
 array([[0.09602067, 0.04342467, 0.11859697],
        [0.097436  , 0.0235364 , 0.09244227],
        [0.09686377, 0.02403225, 0.07342204],
        [0.09745528, 0.04248746, 0.06607691],
        [0.07832605, 0.03747465, 0.04494834],
        [0.08365416, 0.03627936, 0.03586867],
        [0.04965642, 0.01933256, 0.01922481],
        [0.06437046, 0.03524096, 0.01473236],
        [0.00711906, 0.00950123, 0.0289551 ],
        [0.01201405, 0.01200844, 0.01170753],
        [0.        , 0.        ,

In [18]:
attack = np.zeros((len(layer_list),9))
attack_final = np.zeros(9)
for i in range(5):
    attack += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data/VGG-11/layer_cache/attack/"+str(i))
attack = attack/5
for i in range(5):
    path = "./attack/" + str(i)
    ATTACK_DIR = os.path.join(path, "VGG_pgd.npy")
    attack_final += eval_acc_select_list_single_thresholds(model, ATTACK_DIR, 'train', select_list, threshold_list)
attack_final = attack_final/5
attack = np.concatenate((attack,attack_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.84276736 0.6816884  0.61906588 0.58389646 0.61256629 0.48222449
 0.6274117  0.7366276  0.89875718]
Layer 1
accuracy: [0.99426947 0.82153068 0.73869322 0.67117433 0.6137817  0.62998862
 0.70821645 0.33625607 0.6487247 ]
Layer 2
accuracy: [0.97356506 0.77978116 0.70665891 0.65857612 0.58427436 0.38362476
 0.29260505 0.27266572 0.24632921]
Layer 3
accuracy: [0.99120911 0.80134717 0.71989025 0.65631716 0.55959003 0.47703655
 0.63495909 0.78891853 0.90503551]
Layer 4
accuracy: [0.92489077 0.71469413 0.6683913  0.64796393 0.55631886 0.36778432
 0.26195274 0.16346976 0.07886836]
Layer 5
accuracy: [0.92337011 0.71246461 0.66502546 0.64771368 0.55512959 0.36537951
 0.25848766 0.1478698  0.07080342]
Layer 6
accuracy: [0.87470912 0.68694593 0.66206162 0.64791693 0.55511828 0.3642561
 0.2568168  0.14493437 0.06968714]
Layer 7
accuracy: [0.82306942 0.32119591 0.54463502 0.64791693 0.55511828 0.3642561
 0.2568168  0.14724836 0.06991372]
Layer 8
accuracy: [0.34141679 0.17844438 0

In [19]:
compute_stats(attack)

(array([[0.71450721, 0.55956241, 0.75426549],
        [0.85149779, 0.63831489, 0.56439907],
        [0.82000171, 0.54215841, 0.27053333],
        [0.83748218, 0.56431458, 0.77630438],
        [0.7693254 , 0.52402237, 0.16809695],
        [0.76695339, 0.52274093, 0.15905363],
        [0.74123889, 0.52243044, 0.1571461 ],
        [0.56296679, 0.52243044, 0.15799296],
        [0.25882194, 0.41882759, 0.15688191],
        [0.38850482, 0.52243044, 0.15721006],
        [0.73307396, 0.52365196, 0.15715897]]),
 array([[0.09422807, 0.05592469, 0.1114762 ],
        [0.10646854, 0.02415883, 0.16313949],
        [0.11261437, 0.11613176, 0.01895211],
        [0.11367434, 0.07326721, 0.11061843],
        [0.1116137 , 0.11664033, 0.07481547],
        [0.11228612, 0.11751571, 0.07702879],
        [0.09492289, 0.11808821, 0.07688182],
        [0.20529864, 0.11808821, 0.07668018],
        [0.06655168, 0.09700726, 0.07739416],
        [0.24499123, 0.11808821, 0.07680909],
        [0.08351832, 0.11888935,

In [20]:
gauss = np.zeros((len(layer_list),9))
gauss_final = np.zeros(9)
for i in range(10):
    gauss += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data/VGG-11/layer_cache/gauss/"+str(i))
gauss = gauss/10
for i in range(10):
    path = "./noise/" + str(i)
    GAUSS_DIR = os.path.join(path, "gauss.npy")
    gauss_final += eval_acc_select_list_single_thresholds(model, GAUSS_DIR, 'train', select_list, threshold_list)
gauss_final = gauss_final/10
gauss = np.concatenate((gauss,gauss_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.29880977 0.3261873  0.35625997 0.40066883 0.51093067 0.39835445
 0.70902061 0.8211803  0.91901985]
Layer 1
accuracy: [0.99535562 0.82402952 0.74447798 0.6771149  0.62383437 0.62413057
 0.70970483 0.45925364 0.77608041]
Layer 2
accuracy: [0.96885314 0.7639798  0.70905687 0.67728898 0.5874698  0.4041426
 0.31122633 0.22396864 0.11821094]
Layer 3
accuracy: [0.99535562 0.8240282  0.74339537 0.67783242 0.58423968 0.61378384
 0.70902061 0.8211803  0.91901985]
Layer 4
accuracy: [0.99535562 0.8240282  0.74339537 0.67783242 0.5844742  0.39835445
 0.29097939 0.1788197  0.08098015]
Layer 5
accuracy: [0.99535562 0.8240282  0.74339537 0.67783242 0.5844742  0.39835445
 0.29139956 0.2193067  0.29307104]
Layer 6
accuracy: [0.99535562 0.8240282  0.74339537 0.67783242 0.5844742  0.39835445
 0.29097939 0.18172251 0.37216174]
Layer 7
accuracy: [0.99535562 0.8240282  0.74339537 0.67783242 0.5844742  0.39835445
 0.29095669 0.25887736 0.75049387]
Layer 8
accuracy: [0.99535562 0.8240282  

In [21]:
compute_stats(gauss)

(array([[0.32784529, 0.43734548, 0.81640692],
        [0.85456124, 0.64185101, 0.6465085 ],
        [0.81285386, 0.55611071, 0.21782284],
        [0.85425973, 0.62460265, 0.81640029],
        [0.85425973, 0.55355369, 0.18365648],
        [0.85425973, 0.55355369, 0.26755062],
        [0.85425973, 0.55355369, 0.28216834],
        [0.85425973, 0.55355369, 0.43625562],
        [0.85533531, 0.55585321, 0.19635707],
        [0.85440085, 0.55358025, 0.18401142],
        [0.85433706, 0.5545803 , 0.1984554 ]]),
 array([[0.02341614, 0.05275687, 0.08579825],
        [0.10481688, 0.02523927, 0.13995833],
        [0.1103102 , 0.11382087, 0.07882684],
        [0.10506014, 0.03928391, 0.08580654],
        [0.10506014, 0.11617241, 0.08572244],
        [0.10506014, 0.11617241, 0.03680732],
        [0.10506014, 0.11617241, 0.0789448 ],
        [0.10506014, 0.11617241, 0.22853526],
        [0.10394044, 0.11822389, 0.07237351],
        [0.10491131, 0.11619454, 0.08531515],
        [0.1050242 , 0.11692642,

In [22]:
salt = np.zeros((len(layer_list),9))
salt_final = np.zeros(9)
for i in range(10):
    salt += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data/VGG-11/layer_cache/salt/"+str(i))
salt = salt/10
for i in range(10):
    path = "./noise/" + str(i)
    SALT_DIR = os.path.join(path, "salt.npy")
    salt_final += eval_acc_select_list_single_thresholds(model, SALT_DIR, 'train', select_list, threshold_list)
salt_final = salt_final/10
salt = np.concatenate((salt,salt_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.34018085 0.34224179 0.36389521 0.40728404 0.53322822 0.39904571
 0.70902061 0.8211803  0.91901985]
Layer 1
accuracy: [0.04163247 0.18211898 0.25723507 0.32287018 0.44664558 0.39835445
 0.29097939 0.8211803  0.91901985]
Layer 2
accuracy: [0.13778978 0.194856   0.26528891 0.3374279  0.63629312 0.39835445
 0.29097939 0.1788197  0.08098015]
Layer 3
accuracy: [0.98691001 0.79975071 0.70010233 0.63566889 0.6507484  0.48718387
 0.43674415 0.499787   0.46877242]
Layer 4
accuracy: [0.99335768 0.80711063 0.68997501 0.63241117 0.61745635 0.41498401
 0.2952918  0.19153286 0.11380063]
Layer 5
accuracy: [0.99402288 0.82016504 0.72712933 0.67233522 0.61772836 0.44587948
 0.34840969 0.33341438 0.34433989]
Layer 6
accuracy: [0.99535562 0.82506578 0.74235358 0.67689257 0.60209016 0.47713511
 0.47272545 0.59118167 0.85600438]
Layer 7
accuracy: [0.99535562 0.82464028 0.74575192 0.67907103 0.59613335 0.46912492
 0.47248773 0.68038765 0.89087889]
Layer 8
accuracy: [0.99535562 0.82694989

In [23]:
compute_stats(salt)

(array([[0.35038855, 0.44549712, 0.81631257],
        [0.159429  , 0.38863191, 0.67707407],
        [0.19886523, 0.45653756, 0.18359308],
        [0.82903239, 0.58476134, 0.46774445],
        [0.83041663, 0.55521301, 0.19943633],
        [0.84677484, 0.57748439, 0.33975924],
        [0.85391941, 0.58775014, 0.63856353],
        [0.85429498, 0.58284967, 0.67817908],
        [0.81723906, 0.5615856 , 0.41141464],
        [0.83758881, 0.56801808, 0.22747891],
        [0.85203531, 0.567735  , 0.38518559]]),
 array([[0.01148506, 0.06011351, 0.08589695],
        [0.08985391, 0.05039786, 0.27588677],
        [0.05487364, 0.12924165, 0.08579825],
        [0.11851984, 0.07424887, 0.02603483],
        [0.12489617, 0.09867743, 0.07598416],
        [0.11138887, 0.09535297, 0.00629394],
        [0.10545924, 0.08244675, 0.16210788],
        [0.10500202, 0.0837307 , 0.17245748],
        [0.1491501 , 0.06211864, 0.04369037],
        [0.12414528, 0.09606946, 0.06780221],
        [0.10759317, 0.10198743,

In [24]:
move = np.zeros((len(layer_list),9))
move_final = np.zeros(9)
for i in range(10):
    move += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data/VGG-11/layer_cache/move/"+str(i))
move = move/10
for i in range(10):
    path = "./noise/" + str(i)
    MOVE_DIR = os.path.join(path, "move.npy")
    move_final += eval_acc_select_list_single_thresholds(model, MOVE_DIR, 'train', select_list, threshold_list)
move_final = move_final/10
move = np.concatenate((move,move_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.97266094 0.81318402 0.73523475 0.67066848 0.60044571 0.59832667
 0.37064929 0.36520868 0.64144575]
Layer 1
accuracy: [0.18705202 0.26172937 0.30208934 0.34631212 0.43453505 0.402922
 0.30017336 0.82073521 0.91860972]
Layer 2
accuracy: [0.50348983 0.39935368 0.43871823 0.51006246 0.64111854 0.40546048
 0.29286678 0.18914817 0.12720161]
Layer 3
accuracy: [0.71636173 0.5695448  0.56422318 0.52960083 0.64307003 0.4747367
 0.37339674 0.34247011 0.34319852]
Layer 4
accuracy: [0.9951316  0.819088   0.65534125 0.56947644 0.62886362 0.62105507
 0.52100976 0.54279055 0.72229103]
Layer 5
accuracy: [0.9949393  0.81292298 0.68460883 0.62006306 0.64701126 0.63067195
 0.59909219 0.66631078 0.81773058]
Layer 6
accuracy: [0.99535562 0.82627953 0.72183103 0.64561693 0.64522595 0.59370769
 0.56973417 0.64933791 0.82959583]
Layer 7
accuracy: [0.99535562 0.8252611  0.72523091 0.65160307 0.64588605 0.62937929
 0.63159933 0.76422428 0.8932064 ]
Layer 8
accuracy: [0.99535562 0.84709542 0.

In [25]:
compute_stats(move)

(array([[0.83948848, 0.62293957, 0.45915818],
        [0.24843384, 0.39476985, 0.67967044],
        [0.44303197, 0.51853852, 0.20374779],
        [0.61437368, 0.5476913 , 0.35048051],
        [0.82216052, 0.60613319, 0.59110894],
        [0.83064715, 0.62922144, 0.69368623],
        [0.84743128, 0.62533279, 0.68091106],
        [0.84879312, 0.64072029, 0.75750121],
        [0.84405581, 0.6237214 , 0.68132499],
        [0.83562928, 0.62097965, 0.58022829],
        [0.85136485, 0.63021911, 0.64024094]]),
 array([[0.09932454, 0.03349817, 0.12874052],
        [0.04784022, 0.03591271, 0.27204508],
        [0.04297355, 0.09659776, 0.06874534],
        [0.07166652, 0.07128523, 0.01350763],
        [0.13931554, 0.02421042, 0.09043752],
        [0.12674211, 0.00572022, 0.09090913],
        [0.11304468, 0.02513425, 0.10827067],
        [0.1114386 , 0.00977243, 0.10542471],
        [0.12298619, 0.00985561, 0.05659599],
        [0.13140441, 0.01068166, 0.04736704],
        [0.11309757, 0.01614593,

In [26]:
occ = np.zeros((len(layer_list),9))
occ_final = np.zeros(9)
for i in range(10):
    occ += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data/VGG-11/layer_cache/occ/"+str(i))
occ = occ/10
for i in range(10):
    path = "./noise/" + str(i)
    OCC_DIR = os.path.join(path, "occ.npy")
    occ_final += eval_acc_select_list_single_thresholds(model, OCC_DIR, 'train', select_list, threshold_list)
occ_final = occ_final/10
occ = np.concatenate((occ,occ_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.64690176 0.49606253 0.43699263 0.41383987 0.44024468 0.46094612
 0.69836742 0.82033403 0.91901985]
Layer 1
accuracy: [0.75857745 0.52661541 0.38680849 0.37223817 0.4496295  0.55384955
 0.39896175 0.81776481 0.9186117 ]
Layer 2
accuracy: [0.57883104 0.35764082 0.32135391 0.34930816 0.46151245 0.60439442
 0.3584699  0.24613052 0.20368778]
Layer 3
accuracy: [0.91690106 0.69934238 0.53104199 0.42912755 0.4862625  0.63539268
 0.59626642 0.60985901 0.70562879]
Layer 4
accuracy: [0.93235368 0.67274844 0.44648578 0.38495717 0.46245901 0.63553148
 0.6761242  0.74295646 0.85630098]
Layer 5
accuracy: [0.95840382 0.74770276 0.51322957 0.45645747 0.48914767 0.62195457
 0.64990293 0.71122599 0.81017075]
Layer 6
accuracy: [0.99447541 0.85035507 0.68178075 0.5546034  0.49626427 0.61371083
 0.71251706 0.82247948 0.91923298]
Layer 7
accuracy: [0.99468021 0.84958169 0.69040088 0.58476058 0.52019361 0.61638883
 0.71497305 0.82420919 0.91986326]
Layer 8
accuracy: [0.99512824 0.8662227 

In [27]:
compute_stats(occ)

(array([[0.52753363, 0.43907755, 0.81266511],
        [0.55601792, 0.45905535, 0.71299576],
        [0.41800712, 0.47643609, 0.27102613],
        [0.71050778, 0.51942392, 0.64031946],
        [0.67799995, 0.49473512, 0.76198784],
        [0.73390718, 0.52349319, 0.72668443],
        [0.84056182, 0.55739426, 0.81817821],
        [0.84173706, 0.57605051, 0.81993076],
        [0.81724903, 0.52214401, 0.82143374],
        [0.818493  , 0.54919189, 0.81348549],
        [0.85496001, 0.58641346, 0.82092387]]),
 array([[0.08937389, 0.02090942, 0.09023148],
        [0.15210282, 0.08003389, 0.22322752],
        [0.11329629, 0.10851357, 0.06392391],
        [0.15996202, 0.09240171, 0.04933152],
        [0.20298991, 0.10796873, 0.06841323],
        [0.18249526, 0.07676318, 0.06468197],
        [0.12855227, 0.04481148, 0.08461866],
        [0.12598721, 0.0362325 , 0.08367609],
        [0.16859187, 0.07205888, 0.08350868],
        [0.16902894, 0.05456854, 0.08273385],
        [0.11803917, 0.03273767,

In [28]:
SAVE_FILE='VGG-11.pkl'

In [29]:
progress = {"base": base, "attack": attack,"gauss": gauss,"salt": salt,"move": move,"occ": occ}
with open(SAVE_FILE, "wb") as f:
    pickle.dump(progress, f)

In [ ]:
def stats_minmax_from_matrix_dict(matrix_dict, check_num=6): 
    """ 
    Input: 
        matrix_dict: {name: np.ndarray(m, n)}, a dictionary containing 6 matrices 
    Output: 
        stats: { 
            name: { 
                "mean": (m,), 
                "min":  (m,), 
                "max":  (m,) 
            }, ... 
        } 
    For each component i, statistics are computed along axis=1 (over n samples): 
        mean[i] = mean(X[i, :]) 
        min[i]  = min(X[i, :]) 
        max[i]  = max(X[i, :]) 
    """ 
    if check_num is not None and len(matrix_dict) != check_num: 
        raise ValueError(f"Expected {check_num} matrices, but received {len(matrix_dict)}") 
 
    # shape consistency 
    shapes = [np.asarray(v).shape for v in matrix_dict.values()] 
    if len(set(shapes)) != 1: 
        raise ValueError(f"Inconsistent matrix shapes: {shapes}") 
 
    stats = {} 
    for name, X in matrix_dict.items(): 
        X = np.asarray(X) 
        stats[name] = { 
            "mean": X.mean(axis=1), 
            "std":  X.std(axis=1), 
            "min":  X.min(axis=1), 
            "max":  X.max(axis=1), 
        } 
    return stats 

In [32]:
SAVE_FILE='VGG-11.pkl'
with open(SAVE_FILE, "rb") as f:
    progress = pickle.load(f)

In [33]:
mean_var = stats_minmax_from_matrix_dict(progress)

In [34]:
mean_var

{'base': {'mean': array([0.68319099, 0.73127724, 0.73726762, 0.75307033, 0.82473983,
         0.82906122, 0.89581923, 0.90517571, 0.94103249, 0.95672113,
         1.        ]),
  'std': array([0.11574655, 0.09947823, 0.08751772, 0.09565781, 0.07179986,
         0.06688093, 0.04765766, 0.05012544, 0.03707301, 0.02995648,
         0.        ]),
  'min': array([0.52566628, 0.63067496, 0.62137501, 0.6355908 , 0.71684381,
         0.73641367, 0.8304428 , 0.82255379, 0.89132872, 0.90332623,
         1.        ]),
  'max': array([0.88425941, 0.92734973, 0.93063163, 0.95996603, 0.96275817,
         0.96319713, 0.97404626, 0.97463338, 0.9909095 , 0.99152818,
         1.        ])},
 'attack': {'mean': array([0.67611171, 0.68473725, 0.54423115, 0.72603371, 0.48714824,
         0.48291598, 0.47360514, 0.41446339, 0.27817715, 0.35604844,
         0.47129497]),
  'std': array([0.12328748, 0.16631204, 0.24323648, 0.15451791, 0.26735576,
         0.27049439, 0.26014358, 0.23199097, 0.13502975, 0.2222